# The Multiple Regression Model: Complete Guide

## Overview
Welcome to the comprehensive guide on Multiple Linear Regression. Simple linear regression models the relationship between one predictor variable and one response variable. While useful for isolated relationships, real-world systems are multivariable.

Outcomes like Salary, House Prices, and Health Risks emerge from the interaction of many variables simultaneously. A single-variable model ignores most of reality and can lead to dangerous statistical inaccuracies, such as **Omitted Variable Bias**.

In this notebook, we will explore:
1. Why Simple Linear Regression breaks down
2. The core problem of Omitted Variable Bias
3. Building Multiple Linear Regression models
4. The concept of Ceteris Paribus (Holding constant)
5. Diagnostics, Assumptions, and Multicollinearity

In [ ]:
# 1. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# Set display options for better readability
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)

print('Libraries successfully imported! Environment is ready.')

## 2. Data Creation: Synthetic Salary Dataset

To understand multiple regression, we need a realistic multivariable dataset. We will create a synthetic dataset simulating Employee Salaries.

The true population model we are trying to simulate is:
Salary = beta_0 + beta_1 * Education + beta_2 * Experience + epsilon

Where:
- beta_0 (Intercept) = 20k
- beta_1 (Education effect) = 5k per year
- beta_2 (Experience effect) = 3k per year

Crucially, we will design the data so that **Education and Experience are positively correlated**. This correlation is what causes Omitted Variable Bias if we ignore one of them.

In [ ]:
# Generate synthetic data
n_samples = 300

# Education: Normal distribution around 14 years (approx. Associates/Bachelors)
education = np.random.normal(loc=14, scale=2.5, size=n_samples)

# Experience: Correlated with education (people with more education might have specific experience tracks)
# We introduce a positive correlation + some random noise
experience = 0.6 * education + np.random.normal(loc=2, scale=3, size=n_samples)
# Ensure no negative experience
experience = np.maximum(0, experience)

# True Salary Formula (The Hidden Reality)
# Base salary 20k + 5k per year of education + 3k per year of experience + noise
noise = np.random.normal(loc=0, scale=8, size=n_samples)
salary = 20 + 5 * education + 3 * experience + noise

# Combine into a Pandas DataFrame
df_salary = pd.DataFrame({
    'education_years': education,
    'experience_years': experience,
    'salary_k': salary
})

print('Synthetic dataset created with', n_samples, 'records.')

## 3. Initial Data Exploration

Before fitting models, we should always inspect our data. We will check the first few rows, the data types, and basic descriptive statistics.

In [ ]:
# Display the first 5 rows
print('--- First 5 rows of the dataset ---')
print(df_salary.head())
print('\n--- Dataset Information ---')
df_salary.info()
print('\n--- Descriptive Statistics ---')
print(df_salary.describe().round(2))

## 4. Core Concept 1: Omitted Variable Bias

Suppose we are naive and model salary using ONLY education. 

Salary = beta_0 + beta_1 * Education + epsilon

Because Education and Experience are correlated, the `beta_1` coefficient will absorb part of the experience effect. This is called **Omitted Variable Bias**. Let's fit this simple model and see what happens to the Education coefficient.

In [ ]:
# Fit Simple Linear Regression (Salary ~ Education)
simple_model = smf.ols('salary_k ~ education_years', data=df_salary).fit()

# Extract the coefficient for education
simple_edu_coef = simple_model.params['education_years']

print('--- Simple Linear Regression Results ---')
print(f'Estimated Intercept: {simple_model.params["Intercept"]:.2f}')
print(f'Estimated Education Effect: {simple_edu_coef:.2f}k per year')

# Notice: The true effect of education is 5.0k, but the simple model estimates it much higher!
print(f'\nWARNING: The model overestimates the education effect by {(simple_edu_coef - 5.0):.2f}k!')
print('This is Omitted Variable Bias in action. It is absorbing the effect of experience.')

## 5. Visualizing the Biased Simple Regression

Let's visualize this simple relationship. The line will look like a good fit, masking the underlying complexity of the missing variables.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x='education_years', y='salary_k', data=df_salary, 
            scatter_kws={'alpha': 0.6, 'color': 'blue'}, 
            line_kws={'color': 'red', 'linewidth': 2})

plt.title('Simple Regression: Salary vs Education (Biased Fit)', fontsize=14)
plt.xlabel('Education (Years)', fontsize=12)
plt.ylabel('Salary ($K)', fontsize=12)

# Add text to show the distorted slope
plt.text(x=df_salary['education_years'].min(), y=df_salary['salary_k'].max()-10,
         s=f'Apparent Slope: {simple_edu_coef:.2f}\nTrue Slope: 5.00',
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='red'))

plt.tight_layout()
plt.show()

## 6. Core Concept 2: Multiple Linear Regression

To fix the bias, we include multiple predictors simultaneously. 
The model becomes:

Salary = beta_0 + beta_1 * Education + beta_2 * Experience + epsilon

This isolates the effects statistically. Let's fit the multiple regression model.

In [ ]:
# Fit Multiple Linear Regression (Salary ~ Education + Experience)
multi_model = smf.ols('salary_k ~ education_years + experience_years', data=df_salary).fit()

print('--- Multiple Linear Regression Summary ---')
print(multi_model.summary().tables[1]) # Print just the coefficients table for clarity

## 7. Interpretation: Ceteris Paribus

The most important concept in multiple regression is **conditional interpretation**, or *Ceteris Paribus* (all else being equal).

In our multiple model, `beta_1` (Education) means: expected change in Salary for a one-unit increase in Education, **holding Experience constant**.

In [ ]:
# Extract multiple model coefficients
multi_edu_coef = multi_model.params['education_years']
multi_exp_coef = multi_model.params['experience_years']
multi_intercept = multi_model.params['Intercept']

print('--- Coefficient Comparison ---')
print(f'True Education Parameter:      5.00')
print(f'Simple Model (Biased):         {simple_edu_coef:.2f}')
print(f'Multiple Model (Corrected):    {multi_edu_coef:.2f}')
print('-'*40)
print(f'True Experience Parameter:     3.00')
print(f'Multiple Model (Corrected):    {multi_exp_coef:.2f}')

print('\n--- Ceteris Paribus Interpretation ---')
print(f'Holding experience constant, an additional year of education increases salary by {multi_edu_coef:.2f}k.')
print(f'Holding education constant, an additional year of experience increases salary by {multi_exp_coef:.2f}k.')

## 8. Core Concept 3: The Geometry of Multiple Regression

Simple regression fits a **line** in 2D space.
Multiple regression with two predictors fits a **plane** in 3D space.

Let's visualize the 3D regression plane.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot of actual data
ax.scatter(df_salary['education_years'], df_salary['experience_years'], df_salary['salary_k'], 
           c='blue', marker='o', alpha=0.5, label='Actual Data')

# Create a grid of points to plot the plane
x_surf = np.linspace(df_salary['education_years'].min(), df_salary['education_years'].max(), 10)
y_surf = np.linspace(df_salary['experience_years'].min(), df_salary['experience_years'].max(), 10)
x_surf, y_surf = np.meshgrid(x_surf, y_surf)

# Calculate the predicted Z (Salary) for the plane
z_surf = multi_intercept + multi_edu_coef * x_surf + multi_exp_coef * y_surf

# Plot the plane
ax.plot_surface(x_surf, y_surf, z_surf, color='red', alpha=0.3)

ax.set_xlabel('Education (Years)')
ax.set_ylabel('Experience (Years)')
ax.set_zlabel('Salary ($K)')
ax.set_title('Multiple Regression: The Regression Plane in 3D')
plt.show()

## 9. Model Assessment: R-squared vs Adjusted R-squared

Adding variables **always** increases R-squared, even if the variables are useless noise. This leads to **overfitting**.

To combat this, we use **Adjusted R-squared**, which penalizes the model for adding predictors that don't improve the fit more than expected by chance.

In [ ]:
# Add random noise variables to the dataset to demonstrate overfitting
df_noise = df_salary.copy()
for i in range(1, 11):
    df_noise[f'random_noise_{i}'] = np.random.normal(0, 1, n_samples)

# Fit model with noise
formula_noise = 'salary_k ~ education_years + experience_years + ' + ' + '.join([f'random_noise_{i}' for i in range(1, 11)])
noise_model = smf.ols(formula_noise, data=df_noise).fit()

print('--- Model Assessment Comparison ---')
print(f'Original Multiple Model R-squared:     {multi_model.rsquared:.4f}')
print(f'Noise-Added Model R-squared:           {noise_model.rsquared:.4f} (Notice it went UP!)')
print('-'*50)
print(f'Original Multiple Model Adj R-squared: {multi_model.rsquared_adj:.4f}')
print(f'Noise-Added Model Adj R-squared:       {noise_model.rsquared_adj:.4f} (Notice it went DOWN or stayed flat!)')

print('\nKey Takeaway: Never rely solely on R-squared when adding multiple predictors. Use Adjusted R-squared.')

## 10. Checking LINE Assumptions (Residual Analysis)

Multiple regression inherits the LINE assumptions from simple regression:
- **L**inearity
- **I**ndependence
- **N**ormality of residuals
- **E**qual variance (Homoscedasticity)

Let's visually check the residuals of our correct multiple model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Residuals vs Fitted values (Checks Linearity and Equal Variance)
sns.scatterplot(x=multi_model.fittedvalues, y=multi_model.resid, ax=axes[0], alpha=0.7, color='purple')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title('Residuals vs Fitted Values')
axes[0].set_xlabel('Fitted Salary')
axes[0].set_ylabel('Residuals')
axes[0].grid(alpha=0.3)

# Plot 2: Histogram of Residuals (Checks Normality)
sns.histplot(multi_model.resid, kde=True, ax=axes[1], color='teal')
axes[1].set_title('Distribution of Residuals')
axes[1].set_xlabel('Residuals')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. The Danger of Multicollinearity

Multicollinearity occurs when predictor variables are highly correlated with **each other**.
It causes unstable coefficients and inflated standard errors.

Let's add a variable `Age` that is extremely correlated with `Experience` and see how it ruins the model.

In [ ]:
# Create highly collinear variable: Age is just Experience + Education + 18 + tiny noise
df_collinear = df_salary.copy()
df_collinear['age'] = df_collinear['experience_years'] + df_collinear['education_years'] + 18 + np.random.normal(0, 0.5, n_samples)

# Fit model with collinearity
collinear_model = smf.ols('salary_k ~ education_years + experience_years + age', data=df_collinear).fit()

print('--- Model with Multicollinearity (Age added) ---')
print(collinear_model.summary().tables[1])

print('\nNotice how the P-values for experience and age are suddenly high, and coefficients are completely distorted!')
print('This happens because the model cannot distinguish the isolated effect of age vs experience.')

## 12. Diagnosing Multicollinearity

We can diagnose multicollinearity using:
1. Correlation Matrix Heatmap
2. Variance Inflation Factor (VIF) - A VIF > 5 or 10 indicates severe multicollinearity.

In [ ]:
# 1. Correlation Heatmap
plt.figure(figsize=(8, 6))
corr_matrix = df_collinear[['salary_k', 'education_years', 'experience_years', 'age']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Matrix (Notice Age and Experience)', fontsize=14)
plt.show()

# 2. Calculate VIF
print('\n--- Variance Inflation Factor (VIF) ---')
X = df_collinear[['education_years', 'experience_years', 'age']]
X = sm.add_constant(X) # VIF requires constant

vif_data = pd.DataFrame()
vif_data['Feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print(vif_data[vif_data['Feature'] != 'const'].round(2))
print('\nConclusion: Age and Experience have extremely high VIF (> 10), confirming severe multicollinearity.')

## 13. Practice Exercises: House Prices

**Scenario:** You are a data scientist at a real estate agency. You need to model House Prices based on Size (sq ft), Number of Bedrooms, and Age of the house.

First, we will generate the practice dataset.

In [ ]:
# Generate practice data
n_houses = 400
size_sqft = np.random.normal(2000, 500, n_houses)
bedrooms = np.round(size_sqft / 800 + np.random.normal(1, 0.5, n_houses))
bedrooms = np.clip(bedrooms, 1, 6)
age_years = np.random.uniform(0, 50, n_houses)

# True Price: Base 50k + 150 per sqft + 20k per bedroom - 2k per year of age
price_k = 50 + (0.15 * size_sqft) + (20 * bedrooms) - (2 * age_years) + np.random.normal(0, 25, n_houses)

df_houses = pd.DataFrame({
    'size_sqft': size_sqft,
    'bedrooms': bedrooms,
    'age_years': age_years,
    'price_k': price_k
})

print('Practice dataset df_houses created.')
print(df_houses.head())

### Exercise 1
Fit a multiple linear regression model predicting `price_k` using `size_sqft`, `bedrooms`, and `age_years`.
Print the summary of the model.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
house_model = smf.ols('price_k ~ size_sqft + bedrooms + age_years', data=df_houses).fit()

print('--- House Price Model Summary ---')
print(house_model.summary().tables[1])
print(f'\nAdjusted R-squared: {house_model.rsquared_adj:.4f}')

### Exercise 2
Extract the coefficient for `age_years` and interpret what it means using the *Ceteris Paribus* principle. Print the interpretation using an f-string.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
age_coef = house_model.params['age_years']

print('--- Interpretation of Age Coefficient ---')
print(f'The estimated coefficient for Age is {age_coef:.2f}.')
print(f'Interpretation: Holding house size and number of bedrooms constant, ')
print(f'each additional year of age is associated with a decrease in price by {abs(age_coef):.2f}k dollars on average.')

## 14. Visualization Gallery

A great way to understand multiple dimensions in a dataframe is using a pairplot. This visualizes all pairwise relationships and marginal distributions.

In [ ]:
# Pairplot to show all interactions in the Salary dataset
plt.figure(figsize=(10, 8))
sns.pairplot(df_salary, diag_kind='kde', corner=True, 
             plot_kws={'alpha': 0.6, 'color': 'navy'}, 
             diag_kws={'color': 'darkorange'})
plt.suptitle('Pairwise Relationships in Salary Data', y=1.02, fontsize=16)
plt.show()

## 15. Summary and Key Takeaways

- **Single vs Multiple**: Simple regression models Y = f(X). Multiple regression models Y = f(X_1, X_2, ..., X_k).
- **Omitted Variable Bias**: Ignoring important variables that correlate with included predictors forces the model to misattribute their effects, distorting the coefficients.
- **Ceteris Paribus**: In multiple regression, a coefficient beta_j measures the effect of X_j on Y, **holding all other variables constant**.
- **Geometry**: Instead of a line, multiple regression fits a plane (or hyperplane) through multidimensional space.
- **Overfitting and Adj R-squared**: Adding predictors blindly increases standard R-squared. Use Adjusted R-squared to penalize unnecessary complexity.
- **Multicollinearity**: High correlation between predictor variables destabilizes the model. Check using Correlation Matrices and VIF scores.